# MovieLens + IMDb merge
MovieLens `links.csv` maps `movieId` -> `imdbId`. We turn that into IMDb's `tconst` (`tt` + 7-digit zero-padded id)
and join IMDb basics, ratings and crew onto each movie. User ratings are aggregated per movie so we never hold a 32M-row joined table.

In [4]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

DATA = Path("recommender_data")
ML = DATA / "movielens" / "ml-32m"
IMDB = DATA / "imdb"


## 1. Load MovieLens

In [5]:
movies = pd.read_csv(ML / "movies.csv").rename(columns={"genres": "movieLensGenres"})
links = pd.read_csv(ML / "links.csv", dtype={"imdbId": "string"})
tags = pd.read_csv(ML / "tags.csv")
ratings = pd.read_csv(
    ML / "ratings.csv", usecols=["userId", "movieId", "rating"],
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
)
print(movies.shape, links.shape, tags.shape, ratings.shape)
movies.head()

(87585, 3) (87585, 3) (2000072, 4) (32000204, 3)


,movieId,title,movieLensGenres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 2. Load IMDb (only the columns we need)

In [6]:
def read_imdb(name, cols):
    return pd.read_csv(IMDB / name, sep="\t", compression="gzip", usecols=cols,
                       dtype="string", na_values=[r"\N"], quoting=3)

basics = read_imdb("title.basics.tsv.gz",
                   ["tconst", "titleType", "primaryTitle", "startYear", "runtimeMinutes", "genres"])
basics = basics[basics["titleType"].isin(["movie", "tvMovie"])].rename(columns={"genres": "imdbGenres"})
imdb_ratings = read_imdb("title.ratings.tsv.gz", ["tconst", "averageRating", "numVotes"])
crew = read_imdb("title.crew.tsv.gz", ["tconst", "directors", "writers"])
for df in (basics, imdb_ratings, crew):
    print(df.shape)
basics.head()

(915270, 6)
(1718051, 3)
(12844280, 3)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,imdbGenres
8,tt0000009,movie,Miss Jerry,1894,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905,100,<NA>
570,tt0000574,movie,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,1907,90,Drama


## 3. Build the IMDb key and merge the movie-level tables

In [7]:
links["tconst"] = "tt" + links["imdbId"].str.zfill(7)

items = (
    movies
    .merge(links[["movieId", "tconst", "tmdbId"]], on="movieId", how="left", validate="one_to_one")
    .merge(basics.drop(columns="titleType"), on="tconst", how="left", validate="many_to_one")
    .merge(imdb_ratings, on="tconst", how="left", validate="many_to_one")
    .merge(crew, on="tconst", how="left", validate="many_to_one")
)
for c in ["startYear", "runtimeMinutes", "numVotes"]:
    items[c] = pd.to_numeric(items[c], errors="coerce")
items["averageRating"] = pd.to_numeric(items["averageRating"], errors="coerce")

matched = items["primaryTitle"].notna().sum()
print(f"MovieLens movies: {len(items):,} | matched to IMDb: {matched:,} ({matched/len(items):.1%})")

MovieLens movies: 87,585 | matched to IMDb: 77,297 (88.3%)


## 4. Add MovieLens tags and aggregated user ratings

In [8]:
tag_text = (tags.dropna(subset=["tag"]).assign(tag=lambda d: d["tag"].str.lower())
            .groupby("movieId")["tag"].agg(lambda s: ", ".join(sorted(set(s)))).rename("userTags"))
rating_stats = ratings.groupby("movieId")["rating"].agg(mlAvgRating="mean", mlNumRatings="count")

items = (items.merge(tag_text, on="movieId", how="left")
              .merge(rating_stats, on="movieId", how="left"))
items["userTags"] = items["userTags"].fillna("")
items.shape

(87585, 16)

## 5. Demonstration: the merged movie table

In [9]:
cols = ["movieId", "tconst", "title", "primaryTitle", "startYear", "runtimeMinutes",
        "movieLensGenres", "imdbGenres", "averageRating", "numVotes",
        "mlAvgRating", "mlNumRatings", "directors", "userTags"]
items[items["numVotes"] > 100_000].sample(10, random_state=1)[cols]

,movieId,tconst,title,primaryTitle,startYear,runtimeMinutes,movieLensGenres,imdbGenres,averageRating,numVotes,mlAvgRating,mlNumRatings,directors,userTags
14518,76293,tt1279935,Date Night (2010),Date Night,2010,88,Action|Comedy|Romance,"Comedy,Crime,Romance",6.3,181104,3.192765,2861.0,nm0506613,"2010s, absurd, action, aftercreditsstinger, al..."
30107,135143,tt3183660,Fantastic Beasts and Where to Find Them (2016),Fantastic Beasts and Where to Find Them,2016,132,Fantasy,"Adventure,Family,Fantasy",7.2,546353,3.614616,8005.0,nm0946734,"1920's, 1920s, 1926, 20's, 20s, abandoned hous..."
1832,1921,tt0138704,Pi (1998),Pi,1998,84,Drama|Sci-Fi|Thriller,"Drama,Horror,Mystery",7.3,193977,3.729706,11543.0,nm0004716,"3, 6.8-filmaffinity, ambiguous, aronofski, art..."
3653,3753,tt0187393,"Patriot, The (2000)",The Patriot,2000,165,Action|Drama|War,"Action,Drama,War",7.2,317043,3.448051,18624.0,nm0000386,"4k, action, american revolution, american revo..."
22145,113604,tt1355630,If I Stay (2014),If I Stay,2014,107,Drama,"Comedy,Drama,Fantasy",6.7,131978,3.423773,774.0,nm0191712,"based on young adult novel, between life and d..."
2620,2712,tt0120663,Eyes Wide Shut (1999),Eyes Wide Shut,1999,159,Drama|Mystery|Thriller,"Drama,Mystery,Thriller",7.5,440700,3.328104,19727.0,nm0000040,"adapted from:book, alan cumming, amazingness, ..."
15331,80860,tt1055292,Life as We Know It (2010),Life as We Know It,2010,114,Comedy|Romance,"Comedy,Drama,Romance",6.5,146595,3.552168,1476.0,nm0075528,"babies, baby, co-parenting, comedy, drama, emo..."
20119,104218,tt2191701,Grown Ups 2 (2013),Grown Ups 2,2013,101,Comedy,Comedy,5.4,195297,2.499422,865.0,nm0240797,"adam sandler, bare chested male, bare chested ..."
6586,6708,tt0325805,Matchstick Men (2003),Matchstick Men,2003,116,Comedy|Crime|Drama,"Comedy,Crime,Drama",7.2,146883,3.643134,6976.0,nm0000631,"alison lohman, and subtely touching film with ..."
49474,177689,tt5715874,The Killing of a Sacred Deer (2017),The Killing of a Sacred Deer,2017,121,Drama|Horror|Mystery|Thriller,"Drama,Horror,Mystery",7.0,219322,3.675248,2214.0,nm0487166,"2 hours long short film, a24, absurdist, akwar..."


## 6. Demonstration: MovieLens vs IMDb ratings (movies with 1000+ ratings in both)

In [10]:
cmp = items[(items["mlNumRatings"] >= 1000) & (items["numVotes"] >= 1000)].copy()
cmp["mlRating10"] = cmp["mlAvgRating"] * 2  # MovieLens is 0.5-5, IMDb is 1-10
print("Correlation:", round(cmp["mlRating10"].corr(cmp["averageRating"]), 3))
cmp.sort_values("mlNumRatings", ascending=False).head(10)[
    ["title", "startYear", "mlAvgRating", "mlRating10", "averageRating", "numVotes"]]

Correlation: 0.936


,title,startYear,mlAvgRating,mlRating10,averageRating,numVotes
314,"Shawshank Redemption, The (1994)",1994,4.404614,8.809228,9.3,3248784
351,Forrest Gump (1994),1994,4.052744,8.105488,8.8,2541886
292,Pulp Fiction (1994),1994,4.196969,8.393937,8.8,2473454
2480,"Matrix, The (1999)",1999,4.156436,8.312873,8.7,2284117
585,"Silence of the Lambs, The (1991)",1991,4.148367,8.296734,8.6,1755366
257,Star Wars: Episode IV - A New Hope (1977),1977,4.099823,8.199647,8.6,1592260
2867,Fight Club (1999),1999,4.228780,8.457560,8.8,2669574
475,Jurassic Park (1993),1993,3.698623,7.397246,8.2,1201847
522,Schindler's List (1993),1993,4.236990,8.473981,9.0,1607205
4888,"Lord of the Rings: The Fellowship of the Ring,...",2001,4.092134,8.184267,8.9,2242653


## 7. Demonstration: individual ratings joined to IMDb content features

In [11]:
sample = ratings.sample(10, random_state=1).merge(
    items[["movieId", "title", "startYear", "imdbGenres", "runtimeMinutes", "averageRating", "directors"]],
    on="movieId", how="left")
sample

,userId,movieId,rating,title,startYear,imdbGenres,runtimeMinutes,averageRating,directors
0,175746,115617,5.0,Big Hero 6 (2014),2014,"Action,Adventure,Animation",102,7.8,"nm2320658,nm0930261"
1,8591,158,1.0,Casper (1995),1995,"Comedy,Family,Fantasy",100,6.2,"nm1224299,nm0797869"
2,115148,53468,5.0,Fido (2006),2006,"Comedy,Drama,Horror",93,6.7,nm0192933
3,198060,4881,4.0,"Man Who Wasn't There, The (2001)",2001,"Crime,Drama",116,7.5,"nm0001054,nm0001053"
4,120461,46948,3.5,Monster House (2006),2006,"Animation,Comedy,Family",91,6.7,nm1481493
5,95273,3623,4.0,Mission: Impossible II (2000),2000,"Action,Adventure,Thriller",123,6.1,nm0000247
6,26023,109673,3.5,300: Rise of an Empire (2014),2014,"Action,Adventure,Drama",102,6.2,nm1729171
7,184747,2398,4.5,Miracle on 34th Street (1947),1947,"Comedy,Drama,Family",96,7.9,nm0780833
8,79108,535,2.0,Short Cuts (1993),1993,"Comedy,Drama",188,7.6,nm0000265
9,49061,4007,4.5,Wall Street (1987),1987,"Crime,Drama",126,7.3,nm0000231


## 8. Save the merged item table

In [12]:
items.to_csv(DATA / "movie_features.csv", index=False)
print("Saved", DATA / "movie_features.csv", items.shape)

Saved recommender_data\movie_features.csv (87585, 16)


## 9. Cleaning the data

In [13]:
# Build the interaction dataframe from the movie-level features and user ratings
merged_df = ratings.merge(
    items[[
        'movieId', 'title', 'primaryTitle', 'startYear', 'runtimeMinutes',
        'movieLensGenres', 'imdbGenres', 'averageRating', 'numVotes',
        'directors', 'writers', 'userTags', 'mlAvgRating', 'mlNumRatings'
    ]],
    on='movieId',
    how='left',
    validate='many_to_one',
)

# Drop redundant metadata columns and keep only the fields that are useful
columns_to_drop = [
    'tconst',
    'tmdbId',
    'primaryTitle',
    'movieLensGenres',
]
cleaned_df = merged_df.drop(columns=columns_to_drop, errors='ignore').copy()

# Remove rows with missing target ratings and normalize the dtype.
cleaned_df['rating'] = pd.to_numeric(cleaned_df['rating'], errors='coerce')
cleaned_df = cleaned_df.dropna(subset=['rating']).reset_index(drop=True)

# Overview of the merged and cleaned interaction data.
print(f"Merged shape:  {merged_df.shape[0]:,} rows x {merged_df.shape[1]} columns")
print(f"Cleaned shape: {cleaned_df.shape[0]:,} rows x {cleaned_df.shape[1]} columns")
print(f"Rows removed during cleaning: {len(merged_df) - len(cleaned_df):,}")

print("\nColumns and data types:")
print(cleaned_df.dtypes.to_string())

overview = pd.DataFrame({
    "dtype": cleaned_df.dtypes.astype(str),
    "missing": cleaned_df.isna().sum(),
    "missing_%": (cleaned_df.isna().mean() * 100).round(2),
    "unique": cleaned_df.nunique(dropna=True),
})
print("\nMissing values and unique values:")
display(overview.sort_values("missing", ascending=False))

print("\nDuplicate rows:", cleaned_df.duplicated().sum())
print("Unique users:", cleaned_df["userId"].nunique())
print("Unique movies:", cleaned_df["movieId"].nunique())

numeric_columns = cleaned_df.select_dtypes(include="number").columns
print("\nNumeric summary:")
display(cleaned_df[numeric_columns].describe().T.round(2))

print("\nSample of cleaned data:")
display(cleaned_df.head())

,userId,movieId,rating,title,startYear,runtimeMinutes,imdbGenres,averageRating,numVotes,directors,writers,userTags,mlAvgRating,mlNumRatings
0,1,17,4.0,Sense and Sensibility (1995),1995,136,"Drama,Romance",7.7,138388,nm0000487,"nm0000807,nm0000668","1800s, 1810s, 18th century, 19th century, 4k, ...",3.945126,22251.0
1,1,25,1.0,Leaving Las Vegas (1995),1995,111,"Drama,Romance",7.5,147196,nm0001214,"nm3840714,nm0001214","'true art is angst' trope, 90s, adapted from:b...",3.676182,22525.0
2,1,29,2.0,"City of Lost Children, The (Cité des enfants p...",1995,112,"Adventure,Drama,Fantasy",7.4,75415,"nm0001988,nm0000466","nm0012496,nm0000466,nm0001988,nm0491011","2.5, abstract, aging, alternate reality, atmos...",3.929884,9413.0
3,1,30,5.0,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,1995,108,"Crime,Drama,History",7.1,6464,nm0955443,"nm0270819,nm0910841,nm0944481","amazon prime, cabaret, diva, mistress, mob, se...",3.629615,1300.0
4,1,32,5.0,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),1995,129,"Mystery,Sci-Fi,Thriller",8.0,685389,nm0000416,"nm0003408,nm0672459,nm0672466","1990s, 3, abandoned theater, activism, activis...",3.910185,55275.0
